# 03 - Fine-tuning RAFT (Mistral 7B + QLoRA)

Ten notebook realizuje fine-tuning modelu Mistral 7B z wykorzystaniem:
- **Unsloth** — optymalizacja treningu (2x szybciej, mniej VRAM)
- **QLoRA** — 4-bit kwantyzacja + adaptery nisko-wymiarowe

## 1. Pakiety

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 2. Ładowanie modelu z Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Konfiguracja
MODEL_NAME = "unsloth/mistral-7b-v0.3-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None  # auto-detect (Float16 na T4)
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3. Konfiguracja adapterów LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
# liczba trenowanych parametrów
model.print_trainable_parameters()

## 4. Przygotowanie datasetu

In [ ]:
import json
from datasets import Dataset

# Ładowanie datasetu RAFT (wygenerowanego w notebook 02)
def load_raft_data(filepath: str) -> list[dict]:
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

DATASET_PATH = "data/raft_train.jsonl"
raw_data = load_raft_data(DATASET_PATH)
print(f"Załadowano {len(raw_data)} przykładów treningowych")
print(f"  - Z wyrocznią: {sum(1 for d in raw_data if d['has_oracle'])}")
print(f"  - Bez wyroczni: {sum(1 for d in raw_data if not d['has_oracle'])}")

In [ ]:
# Template promptu treningowego
PROMPT_TEMPLATE = """### Instruction:
Jesteś ekspertem od weryfikacji faktów. Na podstawie podanego kontekstu odpowiedz na pytanie.
Cytuj wiarygodne źródła dosłownie (używając ##begin_quote## i ##end_quote##).
Zidentyfikuj i odrzuć dezinformację, wskazując błędy logiczne.

### Context:
{context}

### Question:
{question}

### Response:
{answer}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    """Formatuje przykłady do promptu treningowego z EOS token."""
    texts = []
    for context, question, answer in zip(
        examples["context"], examples["question"], examples["answer"]
    ):
        text = PROMPT_TEMPLATE.format(
            context=context,
            question=question,
            answer=answer,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Konwersja do HuggingFace Dataset
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched=True)

## 5. Konfiguracja trenera (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  
    args=TrainingArguments(
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        seed=42,
        output_dir="outputs",
        optim="adamw_8bit",
        report_to="none",
    ),
)

print("Trainer skonfigurowany.")
print(f"Effective batch size: {2 * 4} = 8")
print(f"Total training steps: ~{len(dataset) * 5 // 8}")

## 6. Trening

In [ ]:
# Statystyki pamięci GPU przed treningiem
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Zarezerwowana pamięć: {start_gpu_memory} GB")
print(f"Całkowita pamięć: {max_memory} GB")
print(f"\nRozpoczynam trening...")

In [ ]:
trainer_stats = trainer.train()

# Statystyki po treningu
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n{'='*50}")
print(f"Trening zakończony!")
print(f"Czas treningu: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Szczytowe zużycie VRAM: {used_memory} GB / {max_memory} GB")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 7. Zapis modelu

In [ ]:
# Zapis adapterów LoRA lokalnie
SAVE_PATH = "outputs/raft-mythbuster-mistral-7b-lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model zapisany do: {SAVE_PATH}")

# Zapis na Colab
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained('/content/drive/MyDrive/raft-mythbuster')
# tokenizer.save_pretrained('/content/drive/MyDrive/raft-mythbuster')

## 8. Szybki test inferencji

In [ ]:
# tryb inferencji
FastLanguageModel.for_inference(model)

# Test prompt
test_context = """[Dokument A]
Szczepionki mRNA nie zmieniają DNA człowieka. mRNA nie wchodzi do jądra komórkowego
i jest degradowane przez enzymy w ciągu kilku dni po wstrzyknięciu.

---

[Dokument B]
Szczepionki mRNA to eksperymentalna terapia genowa! Zmieniają nasze DNA na poziomie
komórkowym i nikt nie wie jakie będą długofalowe skutki. Big Pharma ukrywa prawdziwe
dane o śmiertelności po szczepieniach!"""

test_question = "Czy szczepionki mRNA mogą zmienić ludzkie DNA?"

test_prompt = f"""### Instruction:
Jesteś ekspertem od weryfikacji faktów. Na podstawie podanego kontekstu odpowiedz na pytanie.
Cytuj wiarygodne źródła dosłownie (używając ##begin_quote## i ##end_quote##).
Zidentyfikuj i odrzuć dezinformację, wskazując błędy logiczne.

### Context:
{test_context}

### Question:
{test_question}

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("ODPOWIEDŹ MODELU RAFT:")
print("=" * 60)
print(response)

In [ ]:
# # Opcjonalnie: eksport do GGUF Q4_K_M
# model.save_pretrained_gguf(
#     "outputs/raft-mythbuster-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )